# Notebook 2: stima empirica della media e della varianza. Train/test set.

**Corso:** Introduzione alla Data Science e al Machine Learning

**Docenti** Luca Calatroni, Lorenzo Rosasco

**Assignment:** Gli studenti completino le parti segnate con #TO DO

## Goals
L'obiettivo dell'assignment è
1. Generare campioni i.i.d. da distribuzioni notevoli viste a lezione (Bernoulli, Binomiale, Geometrica, Poisson).
2. Confrontare il valore **teorico** della mean/variance with i corrispondenti valori **empirici** al variare della dimensione dei campioni.
3. Effettuare il calcolo numerose volte (trials) per limitare l'effetto di variabilità del campione.
4. Utilizzare un contesto semplice di predizione per valutare l'errore in fase di **training** rispetto a quello in fase di **test** come funzione della taglia del campion.

## Nota sull'uso dell AI
Potete usare strumenti di AI e LLMs per aiutarvi nella scrittura del compito o nell'utilizzo di librerie specifiche, ma il ragionamento proposto **deve essere il vostro** e dovete essere in grado di spiegarlo in caso di dubbi e in occasione dell'esame.

---

## 0. Setup

Fate girare la seguente cella in cui viene fissato il random seed e le impostazioni per i plot successivi. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# Plot settings (feel free to tweak labels/titles, but avoid changing the core computations)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

---

## 1. Funzioni che calcolano la media/varianza empirica (da completare)

Lavoreremo con variabili aleatorie **discrete**.
Considerati $n$ copie i.i.d della variabile aleatoria X, useremo le seguenti definizioni:
- media empirica:  $ \hat\mu_n = \frac{1}{n}\sum_{i=1}^n X_i $
- varianza empirica:  $ S_n^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i-\hat\mu_n)^2 $

### Task 1.1 — Implementate la funzione che resituisce la media e la varianza empirica 

In [3]:
def emp_mean(x: np.ndarray) -> float:
    #TO DO: Empirical mean of a 1D array.
    return float(np.mean(x))

def emp_var(x: np.ndarray) -> float:
    #TO DO: Empirical variance with Bessel correction (divide by n-1).
    n = len(x)                                                          # numero di campioni nell'array x: ovvero le copie indipendenti e identicamente distribuite
    mu = emp_mean(x)                                                    # mu è la media empirica, richiamo la funzione implementata sopra
    return float(np.sum((x-mu)**2)/(n-1))                               # restituisce la varianza empirica, utilizzando la formula scritta sopra: sommatoria, sottraggo media ad ogni elemento, elevo al quadrato, divido per n-1 e poi converto in float

# Quick sanity check: deve restituire 0.5
print("emp_var([0,1]) =", emp_var(np.array([0,1])))

emp_var([0,1]) = 0.5


### Task 1.2 — Interfaccia per run di processi di sampling/campionamento

Utilizziamo un'interfaccia comune, `sample_dist(name, params, n, rng)` che restituisce un NumPy array 1D di dimensione `n`.

Distribuzioni supportate:
- Bernoulli(p)
- Binomial(n_trials, p)
- Geometric(p)  (support {1,2,...}, numero di tentativi prima del primo successo)
- Poisson(mu)


**Hint:** NumPy già contiene le distribuzioni in questione utilizzando `rng.binomial` (to be used for several n's), `rng.geometric`, e `rng.poisson`.

In [4]:
from typing import Dict, Any

def sample_dist(name: str, params: Dict[str, Any], n: int, rng: np.random.Generator) -> np.ndarray:
    name = name.lower()
    if name == "bernoulli":
        p = params["p"]
        return rng.binomial(n=1, p=p, size=n)
    elif name == "binomial":
        n_trials = params["n_trials"]
        p = params["p"]
        return rng.binomial(n=n_trials, p=p, size=n)
    elif name == "geometric":
        p = params["p"]
        return rng.geometric(p=p, size=n)
    elif name == "poisson":
        mu = params["mu"]
        return rng.poisson(lam=mu, size=n)
    else:
        raise ValueError(f"Unknown distribution: {name}")

---

## 2. Media e varianza teoriche (avete le formule)

Per ogni distribuzione sopra riportate, consideriamo:
- $\mu = \mathbb{E}[X]$
- $\sigma^2 = \mathrm{Var}(X)$

### Task 2.1 — Scrivere per ogni distribuzione la formula della media/varianza teorica

L'output deve essere la copia `(mu, var)`.

In [5]:
def theory_mean_var(name: str, params: Dict[str, Any]) -> tuple[float, float]:
    name = name.lower()
    if name == "bernoulli":
        p = params["p"]
        # TO DO
        mu = p
        var = p*(1-p)
        return mu, var
    elif name == "binomial":
        n_trials = params["n_trials"]
        p = params["p"]
        # TO DO
        mu = n_trials*p
        var = n_trials*p*(1-p)
        return mu, var
    elif name == "geometric":
        p = params["p"]
        # TO DO
        mu = 1/p
        var = (1-p)/(p**2)
        return float(mu), float(var)
    elif name == "poisson":
        mu = params["mu"]
        # TO DO
        var = mu
        return float(mu), float(var)
    else:
        raise ValueError(f"Unknown distribution: {name}")

print("Bernoulli check:", theory_mean_var("bernoulli", {"p": 0.3}))
print("Poisson check:", theory_mean_var("poisson", {"mu": 4}))

Bernoulli check: (0.3, 0.21)
Poisson check: (4.0, 4.0)


---

## 3. Calcolo della media/varianza campionaria all'aumentare del numero dei campioni

Studieremo come $\hat\mu_n$ e $S_n^2$ cambiano all'aumentare del numero dei campioni 'n'.

### Task 3.1 — Un campionamento, molti campioni
Presa una distribuzione a scelta, generate un *singolo campione* contenente 'n_max' copie i.i.d della variabile da studiare e calcolate la media/varianza empirica  utilizzando solamente 'n' valori, aumentando n fino ad n_max.

Il task consiste nel produrre due grafici
1. media empirica al variare del numero dei campioni utilizzati e linea orizzontale con il valore della media teorica
2. varianza empirica al variare del numero dei campioni utilizzati e linea orizzontale con il valore della varianza teorica

Potete utilizzare una scala logaritmica per l'asse x (opzionale, ma utile).

In [ ]:
def prefix_stats(x: np.ndarray, ns: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
   # Return arrays of prefix empirical mean and variance for given ns.
    means = []
    vars_ = []
    for n in ns:
        xn = x[:n]
        # TO DO
        means.append(...)
        vars_.append(...)
    return np.array(means), np.array(vars_)

# ====== Choose distribution and parameters here ======

# TO DO: test with Poisson

dist_name = "poisson"
params = {"mu": 4.0}

n_max = 50_000
x = sample_dist(...,....,...)

ns = np.unique(np.logspace(1, np.log10(n_max), 60).astype(int))  # 10 ... n_max
mhat, vhat = prefix_stats(x, ns)
mu, var = theory_mean_var(dist_name, params)

# Plot mean convergence
plt.figure()
# TO DO 
plt.plot(...)
# TO DO: plot thoeretical mu
plt.axhline(...)
plt.xscale("log")
plt.xlabel("n")
plt.ylabel("value $\hat\mu_n$")
plt.title(f"Empirical mean convergence — {dist_name} {params}")
plt.legend()
plt.show()

# Plot variance convergence
plt.figure()
# TO DO
plt.plot(...)
# TO DO: plot theoretical variance
plt.axhline(...)
plt.xscale("log")
plt.xlabel("n")
plt.ylabel("value $S_n^2$")
plt.title(f"Empirical variance convergence — {dist_name} {params}")
plt.legend()
plt.show()

### Domande

1. La media empirica converge al valore della media teorica? Quanto velocemente?
2. La varianza empirica converge al valore della varianza teorica? Il grafico è più o meno rumoroso che quello della media.
3. Ripetere per una distribuzione diversa e verificare il comportamento.

---

## 4. Variabilità degli stimatori su trials multipli

Il test effettuato sopra potrebbe non essere molto valido, visto che si basa sull'utilizzo di campioni generati in un solo trial. Ripetiamo ora l'esperimento 'T' volte e visualizziamo la distribuzione delle stime.

### Task 4.1 — Distribuzione della media empirica.
Fissate un numero di campioni 'n' e un numero di trials 'T' e plottate l'istogramma della media empirica.

**Suggerimento:** istogramma di $\hat\mu_n$ con una line verticale in corrispondenza della media teorica per vedere la distribuzione attorno al valore di interesse.


In [ ]:
def repeated_estimates(dist_name: str, params: Dict[str, Any], n: int, T: int, rng: np.random.Generator):
    mu_hats = np.empty(T)
    var_hats = np.empty(T)
    for t in range(T):
        # TO DO 
        x = ...
        mu_hats[t] = ...
        var_hats[t] =...
    return mu_hats, var_hats

dist_name = "bernoulli"
params = {"p": 0.3}
n = 50
T = 2000

mu, var = theory_mean_var(dist_name, params)
mu_hats, var_hats = repeated_estimates(dist_name, params, n, T, rng)

plt.figure()
# TO DO: plot histogram of the values across trials, use binning in 40 bins
plt.hist(...)
# TO DO: plot vertical line showing theoretical value
plt.axvline(...)
plt.xlabel(r"$\hat\mu_n$")
plt.ylabel("density")
plt.title(f"Sampling distribution of empirical mean (n={n}, T={T}) — {dist_name} {params}")
plt.legend()
plt.show()

### Task 4.2 — Controllo sul bias

Calcolare la media (su tutti i trials) di:
- medie empiriche ad ogni trial $\hat\mu_n$
- varianza empiriche ad ogni trial $S_n^2$

Confrontatele con il valore teorico.

In [ ]:
print("Average empirical mean:", mu_hats.mean(), " | theory:", mu)
print("Average empirical variance:", var_hats.mean(), " | theory:", var)

### Domande

1. Sulla base delle vostre simulazioni, che tipo di stimatori sono $\hat\mu_n$ e $S_n^2$?
3. Cosa succede se dividete per `n` anziché `n-1` nella formula dello stimatore per la varianza?

---

## 5. Un semplice test di predizione: training VS. test error

### Setup
Considerate il seguente problema di predizione:
> dati i.i.d. $X_1,\dots,X_n$, effettuiamo la predizione su una *nuova* osservazione $X$

Un predittore molto semplice è la media empirica $\hat\mu_n$
Valutiamo la performance utilizzando una loss quadratica (**mean squared error (MSE)**):
- training MSE: average $(X_i - \hat\mu_n)^2$ calcolata sul training set
- test MSE: average $(X_i^{\mathrm{test}} - \hat\mu_n)^2$ calcolato su dati non nel training set

### Task 5.1 — Implementare training/test splitting per il calcolo della media.

**Importante:** per ogni tentativo generare nuovi dati (non nel training!) tramite il richiamo nella funzione del parametro 'rng'

In [ ]:
def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean((a - b) ** 2))

def mean_predictor_train_test_mse(dist_name: str, params: Dict[str, Any], n: int, n_test: int, rng: np.random.Generator):
    x_train = sample_dist(dist_name, params, n, rng)
    mu_hat = emp_mean(x_train)

    # TO DO: training error: average (X_i - mu_hat)^2 on the training set
    train_err = ...

    # test error: average (X_test - mu_hat)^2 on an independent test set
    # TO DO
    x_test = sample_dist(dist_name, params, ...,....)
    test_err = ...
    return train_err, test_err

# Quick smoke test
print(mean_predictor_train_test_mse("poisson", {"mu": 4.0}, n=10, n_test=1000, rng=rng))

(3.0, 4.974)


### Task 5.2 — Valutazione delle curve di errore per trial multipli

Per ogni dimensione del campione 'n', ripetere l'esperimento di training/test 'T' volte e mediare gli errori.
Mostrate su un solo grafico:
- errore medio di training valutato con MSE rispetto ad 'n'
- errore medio di test valutato con MSE rispetto ad 'n'

Confrontate con il valore reale della media della variabile aleatoria studiata.

In [ ]:
def curve_train_test(dist_name: str, params: Dict[str, Any], ns: np.ndarray, n_test: int, T: int, rng: np.random.Generator):
    train = np.zeros_like(ns, dtype=float)
    test = np.zeros_like(ns, dtype=float)
    for i, n in enumerate(ns):
        tr = []
        te = []
        for _ in range(T):
            # TO DO: chiamare (a,b) la coppia training-error, test-error e calcolarla per il trial in questione
            a, b = ....
            tr.append(a); te.append(b)
        train[i] = np.mean(tr)
        test[i] = np.mean(te)
    return train, test

# Test con distribuzione di Poisson

dist_name = "poisson"
params_list = [{"mu": 10.0}]

ns = np.unique(np.logspace(1, 3, 25).astype(int))  # 10..1000
n_test = 5000
T = 200

plt.figure()
for params in params_list:
    train, test = curve_train_test(dist_name, params, ns, n_test, T, rng)
    plt.plot(ns, train, label=f"train MSE {params}")
    plt.plot(ns, test, linestyle="--", label=f"test MSE {params}")
plt.xscale("log")
plt.xlabel("n (training sample size)")
plt.ylabel("MSE")
plt.title(f"Mean predictor: training vs test error — {dist_name}")
plt.legend()
plt.show()

### Domande

1. Perché l'errore MSE in training è più piccolo di quello di test?
2. Quando `n` aumenta, cosa succede alla differenza tra l'errore medio di test e di training?